In [4]:
import os
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
from IPython.display import display


path = kagglehub.dataset_download(
    "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
)
print("Path to dataset files:", path)

DATA_DIR = path
print("Using DATA_DIR:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR))

SMALL_TRANS_FILE = os.path.join(DATA_DIR, "HI-Small_Trans.csv")
SMALL_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Small_accounts.csv")
SMALL_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Small_Patterns.txt")

MEDIUM_TRANS_FILE = os.path.join(DATA_DIR, "HI-Medium_Trans.csv")
MEDIUM_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Medium_accounts.csv")
MEDIUM_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Medium_Patterns.txt")

LARGE_TRANS_FILE = os.path.join(DATA_DIR, "HI-Large_Trans.csv")
LARGE_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Large_accounts.csv")
LARGE_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Large_Patterns.txt")

# Accounts only here (~2.1M rows). HI-Medium_Trans.csv is ~32M rows — do not
# load it with pandas. The single-bank filter below scans it with Polars.
df_bank = pd.read_csv(MEDIUM_ACCOUNTS_FILE)

df_bank.info()

df_bank = df_bank.sort_values(by="Bank ID")
df_bank.head()

# Bank ID -> Bank Name (should be 1 name per ID)
bank_map = (
    df_bank.groupby(["Bank ID", "Bank Name"], as_index=False)
    .size()
    .rename(columns={"size": "n_accounts"})
    .sort_values("Bank ID")
)

print("unique Bank IDs:", df_bank["Bank ID"].nunique())
print("unique Bank Names:", df_bank["Bank Name"].nunique())
print("unique (ID, Name) pairs:", len(bank_map))
print("max names per Bank ID:", df_bank.groupby("Bank ID")["Bank Name"].nunique().max())
print("max IDs per Bank Name:", df_bank.groupby("Bank Name")["Bank ID"].nunique().max())

display(bank_map.head(20))





Path to dataset files: /Users/lachlanbruce/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8
Using DATA_DIR: /Users/lachlanbruce/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8
Files: ['LI-Small_Patterns.txt', 'LI-Medium_Patterns.txt', 'HI-Small_Patterns.txt', 'LI-Medium_Trans.csv', 'HI-Large_Patterns.txt', 'HI-Medium_Patterns.txt', 'HI-Medium_Trans.csv', 'LI-Large_Patterns.txt', 'LI-Medium_accounts.csv', 'HI-Small_Trans.csv', 'HI-Small_accounts.csv', 'LI-Small_accounts.csv', 'LI-Small_Trans.csv', 'HI-Medium_accounts.csv', 'LI-Large_Trans.csv', 'LI-Large_accounts.csv', 'HI-Large_Trans.csv', 'HI-Large_accounts.csv']
<class 'pandas.DataFrame'>
RangeIndex: 2087786 entries, 0 to 2087785
Data columns (total 5 columns):
 #   Column          Dtype
---  ------          -----
 0   Bank Name       str  
 1   Bank ID         int64
 2   Account Number  str  
 3   Entity ID       str  
 4   Entity Name    

,Bank ID,Bank Name,n_accounts
0,0,Hearthstone Bancorp,2853
1,1,France Bank #59,1188
2,2,China Bank #55,1149
3,3,Spain Bank #56,1846
4,4,India Bank #45,2214
5,5,Japan Bank #174,724
6,6,UK Bank #32,1902
7,8,Canada Bank #86,832
8,9,Australia Bank #54,881
9,10,UK Bank #99,1112


In [5]:
# Rank banks by how often their ID shows up on a transfer (From Bank or To Bank).
# Scan with Polars — do not load the ~32M-row CSV into pandas.
print("Counting bank appearances in HI-Medium_Trans.csv ...")
bank_tx = (
    pl.scan_csv(MEDIUM_TRANS_FILE)
    .select(pl.concat_list(["From Bank", "To Bank"]).alias("Bank ID"))
    .explode("Bank ID")
    .group_by("Bank ID")
    .agg(pl.len().alias("n_appearances"))
    .sort("n_appearances", descending=True)
    .collect()
    .to_pandas()
)
bank_tx = bank_tx.merge(
    bank_map[["Bank ID", "Bank Name", "n_accounts"]],
    on="Bank ID",
    how="left",
)
print("banks that appear in at least one transfer:", len(bank_tx))
display(bank_tx.head(20))

Counting bank appearances in HI-Medium_Trans.csv ...


/var/folders/bv/x4byd4t16cb1jl2cjvtqwnyr0000gn/T/ipykernel_70774/344320526.py:7: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("Bank ID")


banks that appear in at least one transfer: 122333


,Bank ID,n_appearances,Bank Name,n_accounts
0,70,2979311,National Bank of Indianapolis,15
1,12,240541,France Bank #62,3574
2,11,235966,Bank of Miami,3378
3,27,233951,UK Bank #1,3738
4,0,221169,Hearthstone Bancorp,2853
5,20,215905,National Bank of Los Angeles,2869
6,112,157492,Mexico Bank #27,2795
7,29,139423,Australia Bank #3,2301
8,14,135548,Spain Bank #32,2070
9,15,134698,Russia Bank #72,2526


## Single-bank view for Q-Detect

IBM AML is a **global** ledger: every bank sees every transfer. Q-Detect monitors **one** bank, so we keep only rows that that bank would actually see:

| `direction` | Meaning |
|---|---|
| `internal` | both accounts sit at the monitored bank |
| `outbound` | from a customer of this bank to an account at another bank |
| `inbound` | from another bank into a customer of this bank |

Filter on **`Bank ID`**, not `Bank Name`. One ID always has one name, but names like `Savings Bank of Chicago` are reused across many IDs. Numbered names (`France Bank #59`) are unique, and the `#N` is **not** the ID (`France Bank #59` is ID `1`; `France Bank #62` is ID `12`).

Default below is **Bank ID `0` / Hearthstone Bancorp** (unique name, ~210k HI-Medium transfers, 552 laundering labels). Change `MAIN_BANK_ID` to try another row from `bank_map`.


In [ ]:
# Key off Bank ID. Examples from bank_map: 0 = Hearthstone Bancorp,
# 1 = France Bank #59, 11 = Bank of Miami, 12 = France Bank #62.
MAIN_BANK_ID = 0

main_row = bank_map.loc[bank_map["Bank ID"] == MAIN_BANK_ID]
if main_row.empty:
    raise ValueError(f"Bank ID {MAIN_BANK_ID} is not in HI-Medium_accounts.csv")

MAIN_BANK_NAME = main_row["Bank Name"].iloc[0]
print(f"Monitoring Bank ID {MAIN_BANK_ID} ({MAIN_BANK_NAME})")
print("accounts at this bank:", int(main_row["n_accounts"].iloc[0]))

df_bank_main = df_bank.loc[df_bank["Bank ID"] == MAIN_BANK_ID].copy()
display(df_bank_main.head())

In [ ]:
# Scan HI-Medium and keep transfers that touch the monitored bank.
# Counterparties at other banks stay in the table — a real bank still sees
# the other side of a wire, just not that other bank's internal book.
tx_lf = pl.scan_csv(MEDIUM_TRANS_FILE)
cols = tx_lf.collect_schema().names()
if "Account_duplicated_0" in cols:
    tx_lf = tx_lf.rename({"Account": "From Account", "Account_duplicated_0": "To Account"})
elif "Account.1" in cols:
    tx_lf = tx_lf.rename({"Account": "From Account", "Account.1": "To Account"})
else:
    tx_lf = tx_lf.rename({"Account": "From Account"})

df = (
    tx_lf.filter(
        (pl.col("From Bank") == MAIN_BANK_ID) | (pl.col("To Bank") == MAIN_BANK_ID)
    )
    .with_columns(
        pl.when(
            (pl.col("From Bank") == MAIN_BANK_ID) & (pl.col("To Bank") == MAIN_BANK_ID)
        )
        .then(pl.lit("internal"))
        .when(pl.col("From Bank") == MAIN_BANK_ID)
        .then(pl.lit("outbound"))
        .otherwise(pl.lit("inbound"))
        .alias("direction")
    )
    .collect()
    .to_pandas()
)

print(f"rows touching {MAIN_BANK_NAME}: {len(df):,}")
print(f"laundering labels: {int(df['Is Laundering'].sum()):,}  "
      f"({df['Is Laundering'].mean():.4%} of this bank's book)")
display(df.head())

In [ ]:
# What this bank actually sees: internal vs money leaving vs money arriving.
by_dir = (
    df.groupby("direction")
    .agg(
        n_tx=("direction", "size"),
        n_laundering=("Is Laundering", "sum"),
        n_from_banks=("From Bank", "nunique"),
        n_to_banks=("To Bank", "nunique"),
    )
)
by_dir["laundering_rate"] = by_dir["n_laundering"] / by_dir["n_tx"]
display(by_dir.reindex(["internal", "outbound", "inbound"]))

print("other banks on outbound wires:", df.loc[df["direction"] == "outbound", "To Bank"].nunique())
print("other banks on inbound wires:", df.loc[df["direction"] == "inbound", "From Bank"].nunique())

ax = df["direction"].value_counts().reindex(["internal", "outbound", "inbound"]).plot(
    kind="bar", rot=0, title=f"Transfers visible to {MAIN_BANK_NAME}"
)
ax.set_ylabel("count")
plt.show()